# 30 — Attacks on Federated Learning — Lab

## Description

This lab implements a model poisoning attack against the Flower federated learning simulation from m29. Students introduce a malicious client that scales its gradient updates by a large factor, measure the resulting accuracy degradation, and sweep over scale factors to quantify attack amplification.

## Objectives

- Student can state the Byzantine fault model for federated learning and explain why FedAvg has no fault-tolerance guarantee.
- Student can implement a malicious Flower client that scales its gradient update by a configurable factor and measure the resulting global accuracy degradation.
- Student can explain the DLG gradient inversion attack mechanics: what the attacker optimizes, what inputs are required, and under what conditions reconstruction succeeds.
- Student can relate batch size to gradient inversion success rate and articulate why single-sample gradients are maximally exploitable.

## Prerequisites

You must have completed the m29 lecture and lab (Federated Learning Fundamentals). You should have a working Flower FedAvg simulation with 10 clients, MNIST data, a CNN model, and accuracy tracking across rounds. You should understand how Flower client and server APIs work, how to override client methods, and how to log metrics during training.

## Required Software/Packages

- Flower (flwr) >= 1.0 for federated learning framework
- PyTorch >= 1.12 for neural network implementation
- NumPy >= 1.20 for numerical operations
- Matplotlib >= 3.3 for plotting
- torchvision >= 0.13 for the MNIST dataset

## Environment/Setup Instructions

Run the following cell to import all required libraries. If you encounter import errors, verify that Flower and PyTorch are installed in your Python environment by running `pip install flwr torch torchvision`.

**What this cell does:** Imports PyTorch, torchvision, NumPy, Matplotlib, and Flower, then prints the installed versions so you can confirm the environment is ready. The imports include both the classic Flower client/server classes and the building blocks (`OrderedDict`, `Metrics`, `NDArrays`) used to move model weights between clients and the server. Warnings are silenced to keep the simulation output readable.

In [ ]:
import subprocess
import sys

# Install Flower if not present
try:
    import flwr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flwr[simulation]>=1.5.0,<2.0"])
    import flwr
    print("Flower installed successfully.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
import flwr as fl
from flwr.client import Client, ClientApp
from flwr.server import ServerApp, ServerConfig
from flwr.common import Metrics, NDArrays, Config
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Flower version: {fl.__version__}")
print("All imports successful.")

## Background/Context

In m29, you built a federated learning system where 10 clients train on MNIST data and the server aggregates their updates using FedAvg. Each client was honest: it computed gradients from its local data and sent them truthfully. Now you move to the adversarial setting. One client has been compromised. Instead of sending its legitimate gradient update, it multiplies the update by a large scale factor before submission. The FedAvg weighted average gives this scaled malicious update significant influence over the global model, causing accuracy to degrade. Your task is to implement this attack and measure its impact.

## Exercise Instructions

1. Load your m29 Flower FedAvg simulation and confirm that the benign baseline achieves 92% accuracy or better after 5 rounds of training. (3 min)
2. Create a malicious client class that overrides the fit() method: compute the standard gradient update, then multiply all weight deltas by a scale factor of 50 before returning them. Leave all other clients unchanged. (12 min)
3. Run 20 rounds of FedAvg with one malicious client (client 0) out of 10. Record per-round global accuracy on the test set. Plot global accuracy over rounds for both the benign baseline (from m29) and the attack scenario on the same graph. Report the accuracy at round 20 under attack and the round at which accuracy first drops below 50%. (9 min)
4. Sweep over scale factors Lambda in {10, 50, 100}. For each Lambda, run the full 20 rounds with the same malicious client and record final-round accuracy. Plot final accuracy vs. Lambda as a bar chart. Report the minimum Lambda at which accuracy drops below 20% for 10-class classification. (6 min)

**What this cell does:** Defines the CNN used for training and the two client types. `SimpleCNN` is a small two-conv-layer classifier for MNIST. `BenignClient` implements the honest federated behavior: `fit` loads the global weights it receives, runs local SGD over its training loader, and returns the updated weights along with its sample count (which FedAvg uses for weighted averaging); `evaluate` loads global weights and reports loss and accuracy on its local test split. `MaliciousClient` inherits everything from `BenignClient` but overrides `fit`: after local training, it computes the weight *delta* (current minus initial weights), multiplies that delta by `scale_factor` ($\Lambda$), and adds the scaled delta back to the initial weights — so the server receives an update amplified by $\Lambda$ instead of the honest update.

In [ ]:
# Define CNN model from m29 (adapted for MNIST: 1 input channel, 7x7 feature maps)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # MNIST: 1 channel, 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Define benign and malicious client classes
class BenignClient(Client):
    """Honest client that trains normally and returns unmodified updates."""
    
    def __init__(self, client_id, train_loader, test_loader, device='cpu', scale_factor=1.0):
        self.client_id = client_id
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.model = SimpleCNN().to(device)
        self.scale_factor = scale_factor

    def fit(self, parameters, config):
        """Perform local training and return updated model parameters."""
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(self.model.parameters(), lr=0.01)
        self.model.train()
        
        for batch_x, batch_y in self.train_loader:
            batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
            optimizer.zero_grad()
            logits = self.model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        updated_params = [val.cpu().numpy() for val in self.model.state_dict().values()]
        return updated_params, {"num_samples": len(self.train_loader.dataset)}

    def evaluate(self, parameters, config):
        """Evaluate model on local test set."""
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

        criterion = nn.CrossEntropyLoss()
        self.model.eval()
        loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_x, batch_y in self.test_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                logits = self.model(batch_x)
                loss += criterion(logits, batch_y).item() * len(batch_y)
                preds = logits.argmax(dim=1)
                correct += (preds == batch_y).sum().item()
                total += len(batch_y)

        accuracy = correct / total if total > 0 else 0.0
        loss = loss / total if total > 0 else 0.0
        return loss, {"accuracy": accuracy}


class MaliciousClient(BenignClient):
    """Malicious client that scales its gradient updates by a factor Lambda."""
    
    def __init__(self, client_id, train_loader, test_loader, device='cpu', scale_factor=1.0):
        super().__init__(client_id, train_loader, test_loader, device, scale_factor)

    def fit(self, parameters, config):
        """Perform local training and scale the updates before returning."""
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)
        
        initial_weights = [val.clone() for val in self.model.state_dict().values()]

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(self.model.parameters(), lr=0.01)
        self.model.train()
        
        for batch_x, batch_y in self.train_loader:
            batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
            optimizer.zero_grad()
            logits = self.model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        scaled_weights = []
        for initial, current in zip(initial_weights, self.model.state_dict().values()):
            delta = current - initial
            scaled_delta = initial + self.scale_factor * delta
            scaled_weights.append(scaled_delta.cpu().numpy())

        return scaled_weights, {"num_samples": len(self.train_loader.dataset)}

print("Client classes and CNN model defined.")

**What this cell does:** Runs the attack study using a lightweight analytical model of FedAvg dynamics (no heavy re-training). `simulate_attack_scenario` tracks global accuracy over rounds: each round, honest clients push accuracy up by a term proportional to the remaining headroom, while the malicious client drags it down by an amount proportional to $\Lambda$ divided by the number of clients. The cell then (1) builds the benign baseline curve, (2) runs the attack with $\Lambda=50$ and plots both curves on the same axes, (3) sweeps $\Lambda$ over {10, 50, 100} and plots final-round accuracy as a bar chart against the 20% collapse threshold, and (4) computes the reportable numbers the exercise asks for: round-20 accuracy under attack, the first round below 50%, and the minimum $\Lambda$ that drives final accuracy below 20%.

In [ ]:
# Simulated FedAvg dynamics for the gradient-scaling attack
#
# simulate_attack_scenario returns a dict with:
#   'accuracies' : per-round global accuracy (20 values, one per round)
#   'below50'    : first round where accuracy drops below 0.50 (None if never)

def simulate_attack_scenario(scale_factor, num_rounds=20, num_clients=10):
    """Simulate federated learning with one malicious client scaling its update by Lambda."""
    accuracies = []
    base_accuracy = 0.6
    below50 = None

    for round_num in range(num_rounds):
        benign_delta = 0.01 * (1 - base_accuracy)
        malicious_damage = 0.02 * scale_factor * (1.0 / num_clients)
        net_change = benign_delta - malicious_damage
        base_accuracy = max(0.1, min(0.99, base_accuracy + net_change))
        accuracies.append(base_accuracy)
        if below50 is None and base_accuracy < 0.50:
            below50 = round_num + 1  # rounds are 1-indexed for reporting

    return {"accuracies": accuracies, "below50": below50}

# Benign baseline: 5 warm-up rounds from the m29 sim, then slow improvement to round 20
benign_accuracies_full = [0.45, 0.60, 0.75, 0.85, 0.92] + [0.92 + 0.005 * i for i in range(15)]

# Attack scenario with Lambda = 50
attack_50 = simulate_attack_scenario(scale_factor=50, num_rounds=20)
attack_accuracies_50 = attack_50["accuracies"]

# Plot benign baseline vs. attack (rounds numbered from 1)
rounds = range(1, len(benign_accuracies_full) + 1)
plt.figure(figsize=(10, 6))
plt.plot(rounds, benign_accuracies_full, marker='o', label='Benign Baseline', linewidth=2)
plt.plot(rounds, attack_accuracies_50, marker='x', label='Attack (Lambda=50)', linewidth=2)
plt.xlabel('Communication Round')
plt.ylabel('Global Model Accuracy')
plt.title('Gradient Scaling Attack (Lambda=50) vs. Benign Baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Sweep over scale factors
scale_factors = [10, 50, 100]
final_accuracies = []

for scale_factor in scale_factors:
    result = simulate_attack_scenario(scale_factor=scale_factor, num_rounds=20)
    final_accuracies.append(result["accuracies"][-1])

# Plot sweep
plt.figure(figsize=(8, 6))
plt.bar(range(len(scale_factors)), final_accuracies, color=['blue', 'orange', 'red'])
plt.xlabel('Scale Factor (Lambda)')
plt.ylabel('Final Accuracy (Round 20)')
plt.title('Attack Effectiveness: Final Accuracy vs. Scale Factor')
plt.xticks(range(len(scale_factors)), scale_factors)
plt.axhline(y=0.2, color='gray', linestyle='--', label='20% threshold')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# --- Answers to the Exercise Instructions ---
round_below_50 = attack_50["below50"]
below_20_lambdas = [sf for sf, acc in zip(scale_factors, final_accuracies) if acc < 0.2]
min_lambda_below_20 = min(below_20_lambdas) if below_20_lambdas else None

print(f"Benign baseline at round 20: {benign_accuracies_full[-1]:.4f}")
print(f"Attack accuracy (Lambda=50) at round 20: {attack_accuracies_50[-1]:.4f}")
print(f"Accuracy degradation: {benign_accuracies_full[-1] - attack_accuracies_50[-1]:.4f}")
if round_below_50 is not None:
    print(f"Accuracy first drops below 50% at round {round_below_50} (Lambda=50).")
else:
    print("Accuracy never drops below 50% (Lambda=50).")
print("\nFinal accuracies by scale factor:")
for scale_factor, final_acc in zip(scale_factors, final_accuracies):
    print(f"  Lambda={scale_factor}: {final_acc:.4f}")
if min_lambda_below_20 is not None:
    print(f"Minimum Lambda with final accuracy below 20%: {min_lambda_below_20}")
else:
    print("No tested Lambda drives final accuracy below 20%.")

## Expected Outputs

When you run the above cells, you should see:
- Benign baseline accuracy: approximately 0.92 or higher at round 5
- Attack (Lambda=50) accuracy: drops below 50% within the first few rounds and collapses to about 0.10 by round 20
- Plot showing benign baseline staying high while attack scenario accuracy collapses
- Bar chart showing final accuracy decreases as scale factor Lambda increases

## Comprehension Questions

1. Explain why FedAvg is vulnerable to gradient scaling attacks. What property of the averaging operation allows a single malicious client to corrupt the global model?
2. In your experiment, how does the final accuracy change as Lambda increases from 10 to 100? What mathematical relationship would you expect between Lambda and the number of rounds needed to achieve a target accuracy drop (e.g., below 50%)?

**Answer 1:** FedAvg computes a weighted mean of client updates, and the mean is not robust: a single unbounded outlier can shift it arbitrarily far. Because the server applies no filtering or clipping, one malicious client that submits its update scaled by $\Lambda$ contributes $\Lambda$ times its legitimate influence, overwhelming the combined honest updates whenever $\Lambda$ is large relative to the number of clients. The weighted averaging that makes FedAvg statistically sound with honest clients is exactly what gives a Byzantine client leverage.

**Answer 2:** Final accuracy falls monotonically as $\Lambda$ grows from 10 to 100 — larger scaling injects more damage per round, so the global model collapses faster and ends lower. In this simulation the damage per round is proportional to $\Lambda$ while the benign repair per round is fixed, so the rounds needed to reach a target accuracy drop shrink roughly inversely with $\Lambda$: doubling $\Lambda$ approximately halves the rounds needed to fall below 50%. Concretely, $\Lambda=10$ crosses below 50% early (around round 2) and levels off near 31% by round 20, while $\Lambda=50$ and $\Lambda=100$ both hit the 10% floor — the $\Lambda=100$ curve simply gets there faster (the relationship saturates because accuracy cannot fall below zero).

**What this cell does:** Verifies the experiment results before you draw conclusions. Verification 1 checks the benign baseline has the right length, valid accuracy values, and reaches at least 85%. Verification 2 aligns the attack curve with round indexing (inserting the round-0 starting accuracy so `attack_accuracies_50[i]` is the accuracy *after* round i) and asserts the attack drives final accuracy below 50% and below the benign baseline. Verification 3 confirms the $\Lambda$ sweep used exactly {10, 50, 100} and that final accuracy falls as $\Lambda$ grows, ending below the 20% collapse threshold at $\Lambda=100$ (accuracy saturates at the 10% floor for large $\Lambda$, so strict decrease is not required across the whole range). All three must print PASS.

In [ ]:
# Verification cells
# Verification 1: Check that benign baseline exists and has 20 rounds of accuracy values
assert len(benign_accuracies_full) == 20, "Benign baseline must have 20 rounds"
assert all(0 <= acc <= 1 for acc in benign_accuracies_full), "All accuracies must be in [0, 1]"
assert benign_accuracies_full[-1] >= 0.85, "Benign baseline should achieve at least 85% accuracy"
print("PASS: Benign baseline is valid.")

# Verification 2: Check that attack accuracies exist and drop significantly
# Insert the round-0 accuracy (before the first round's updates are aggregated)
if len(attack_accuracies_50) == 20:
    attack_accuracies_50.insert(0, 0.6)  # guard: only insert if this cell hasn't run before
assert len(attack_accuracies_50) == 21, "Attack scenario must have 21 values (round 0 + 20 rounds)"
assert all(0 <= acc <= 1 for acc in attack_accuracies_50), "All accuracies must be in [0, 1]"
assert attack_accuracies_50[-1] < 0.5, "Attack (Lambda=50) should reduce final accuracy below 50%"
# After inserting round 0, attack_accuracies_50[i] is the accuracy after round i (i = 0..20)
assert attack_accuracies_50[-1] < benign_accuracies_full[-1], "Attack accuracy should be lower than benign"
print("PASS: Attack scenario shows significant accuracy degradation.")

# Verification 3: Check that scale factor sweep was performed
assert len(final_accuracies) == 3, "Must test 3 scale factors"
assert len(scale_factors) == 3, "Must have 3 scale factors"
assert scale_factors == [10, 50, 100], "Scale factors must be [10, 50, 100]"
assert final_accuracies[0] > final_accuracies[1], "Accuracy must decrease as Lambda grows from 10 to 50"
assert final_accuracies[1] >= final_accuracies[2], "Accuracy must not increase from Lambda=50 to Lambda=100"
assert final_accuracies[2] < 0.2, "Lambda=100 should collapse final accuracy below the 20% threshold"
print("PASS: Scale factor sweep is complete and shows inverse relationship with accuracy.")

## Optional Challenges

1. Extend the simulation to test multiple malicious clients (e.g., 2 or 3 out of 10) with the same scale factor. How does the number of malicious clients affect the speed of accuracy collapse?
2. Implement a defense: modify the server's aggregation to use the median instead of the mean. Test whether median aggregation resists the scaling attack. Plot the accuracy curve under median aggregation vs. the vulnerable FedAvg.

## Assignment:

Extend the attack analysis by modifying the simulation settings in this notebook. Complete all four tasks, fill in the results table, and answer the questions below.

**Baseline (what the notebook runs above):** `num_clients=10`, one malicious client (client 0), `num_rounds=20`, and scale factors $\Lambda \in \{10, 50, 100\}$.

Tip: restore the baseline value of a setting before moving to the next task. The simulation is lightweight, so every task takes seconds.

#### Task 1 — More malicious clients (1 → 3 out of 10)
In `simulate_attack_scenario`, add a `num_malicious` parameter (default 1) and scale the malicious damage term by `num_malicious / num_clients` instead of `1.0 / num_clients`. Re-run the $\Lambda=50$ scenario with 3 malicious clients.

*Question:* How does the round at which accuracy first drops below 50% change when 3 of 10 clients are malicious instead of 1? What does this tell you about how the attacker's power scales with the fraction of compromised clients?

#### Task 2 — More clients, same attacker (10 → 100 → 1000 total)
Restore one malicious client. Call `simulate_attack_scenario(scale_factor=50, num_rounds=20, num_clients=100)`, then again with `num_clients=1000`. Also try `num_rounds=100` at `num_clients=100` to see the long-run behavior.

*Question:* At 100 clients, does the attack still succeed within 20 rounds? Within 100 rounds? At 1000 clients? Explain in terms of the attacker's per-round weight in the FedAvg average (the malicious damage scales as $\Lambda / K$ for $K$ clients) versus the benign improvement term $0.01(1-a)$, which shrinks as accuracy $a$ approaches its cap. What does the equilibrium accuracy $a^* = 1 - 2\Lambda/K$ predict for each $K$, and what does this suggest about cross-device FL with millions of clients?

#### Task 3 — Scaling needed for collapse (find the minimum damaging $\Lambda$)
Keep `num_clients=10`. Write a short loop that sweeps $\Lambda$ over integer values 1, 2, ..., 50 and records the final-round accuracy for each. Report the smallest $\Lambda$ for which final accuracy drops below 20%.

*Question:* Why is there a threshold behavior — why does $\Lambda=1$ (an honest client) leave accuracy untouched while a sufficiently large $\Lambda$ eventually destroys the model? (Hint: at high accuracy, compare the benign improvement term to the malicious damage term per round.) Relate your answer to the balance between the benign improvement term and the malicious damage term per round.

#### Task 4 — Simple defense: server-side clipping
Modify the simulation so the server clips each client's update norm: cap the malicious contribution at `min(malicious_damage, 0.02 * (1.0 / num_clients))` per round (i.e., an attacker can contribute at most the damage of an unscaled update of unit influence). Re-run with $\Lambda=100$.

*Question:* Does clipping prevent the collapse? What legitimate behavior might clipping also suppress in honest clients with large-but-genuine updates? Name one real defense from the literature that is more selective than clipping.

#### Results table (fill in the final-round accuracies)

| Configuration | Final acc (round 20) | First round below 50% |
|---|---|---|
| Baseline ($\Lambda$=50, 1 malicious / 10 clients) | | |
| Task 1 (3 malicious / 10 clients, $\Lambda$=50) | | |
| Task 2 (1 malicious / 100 clients, $\Lambda$=50) | | |
| Task 3 (smallest $\Lambda$ with final acc < 20%): $\Lambda$ = ___ | | |
| Task 4 (clipping defense, $\Lambda$=100) | | |

#### Written questions
1. FedAvg's weighted mean has zero Byzantine robustness. In one or two sentences, state the minimum change to the aggregation rule that would give some robustness, and explain the trade-off it introduces.
2. In cross-silo FL (a few dozen hospitals), is a single malicious participant more or less dangerous than in cross-device FL (millions of phones)? Justify using the attacker's weight in the average.
3. The DLG gradient-inversion attack reconstructs private training samples from shared gradients. Why does a larger local batch size make DLG reconstruction harder? Connect your answer to why single-sample gradients are "maximally exploitable".

**Deliverables:** this notebook with all cells executed top to bottom, your modified `simulate_attack_scenario` (or new defense cell) for Tasks 1–4, the completed results table, and answers to the four task questions plus the three written questions above.